In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import os
import numpy as np
from sklearn.manifold import TSNE
%matplotlib inline

## Configuration

In [ ]:
dataset_name = "MNIST" # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
model_type = "Fine_Tuned" # Fine_Tuned | Base
layer_norm = "Pre_Post_Layer_Norm" # Pre_Post_Layer_Norm | Post_Layer_Norm
num_classes = 10

results_path = f"../Data/Embedding_Captures/{dataset_name}/{domain}/{layer_norm}/Entire_Transformation_Matrix_W"
indices = [i for i in range(12)]

## File Prepping

In [ ]:
model_type = "Fine_Tuned"
fine_tuned_path = f"{results_path}/{model_type}"
results_fine_tuned = [] # File Loading

try:
    for filename in os.listdir(fine_tuned_path):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(fine_tuned_path, filename)
        if os.path.isfile(file_path):
            results_fine_tuned.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

results_fine_tuned = [pd.read_json(i) for i in results_fine_tuned]
results_fine_tuned = sorted(results_fine_tuned, key=lambda df: df["Label"][0])

In [ ]:
model_type = "Base"
base_path = f"{results_path}/{model_type}"
results_base = [] # File Loading

try:
    for filename in os.listdir(base_path):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(base_path, filename)
        if os.path.isfile(file_path):
            results_base.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

results_base = [pd.read_json(i) for i in results_base]
results_base = sorted(results_base, key=lambda df: df["Label"][0])

In [ ]:
fine_tuned_embeddings = {}
base_embeddings = {}
for i in range(num_classes):
   fine_tuned_embeddings[i] = np.array([np.array(embed) for embed in results_fine_tuned[i]["W"]])
   base_embeddings[i] = np.array([np.array(embed) for embed in results_base[0]["W"]])

In [ ]:
task_matrix = []
task_matrix_path = "../Data/Class_Specific_B_F/MNIST/Base_Fine_Tuned/Entire_Transformation_Matrix_W"

try:
    for filename in os.listdir(task_matrix_path):
        if filename in ["Standard_48000_Results_All_Classes.json"]:
            file_path = os.path.join(task_matrix_path, filename)
            if os.path.isfile(file_path):
                task_matrix.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{task_matrix_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

task_matrix = np.array([pd.read_json(i) for i in task_matrix][0]["W"][11])

## Augmentations

In [ ]:
# task_matrix (768, 768)
# fine_tuned_embeddings (10, 12, 768) 
# base_embeddings (10, 12, 768)

In [ ]:
augmented_embeddings = {}
for i in range(num_classes):
    augmented_embeddings[i] = []
    for j in indices:
        augmented_embeddings[i].append(np.array(base_embeddings[i][j] @ task_matrix))
    augmented_embeddings[i] = np.array(augmented_embeddings[i])

## Graphs

### Actual

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embeddings_2d = tsne.fit_transform(fine_tuned_embeddings[0])
t_2d = tsne.fit_transform(augmented_embeddings[0])

plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color="blue")
plt.scatter(t_2d[:, 0], t_2d[:, 1], color="green")
for i, (x, y) in enumerate(embeddings_2d):
    plt.text(x + 0.5, y, f"Layer {i}", fontsize=9)
for i, (x, y) in enumerate(t_2d):
    plt.text(x + 0.5, y, f"Layer {i}", fontsize=9)

plt.title("t-SNE of 12 Layer Embeddings for digit 0")
plt.show()

In [ ]:
condensed_base_last = []
condensed_fine_tuned_last = []
condensed_augmented_last = []

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42)

for i in range(num_classes):
    base = tsne.fit_transform(base_embeddings[i])
    fine_tuned = tsne.fit_transform(fine_tuned_embeddings[i])
    augmented = tsne.fit_transform(augmented_embeddings[i])
    condensed_base_last.append((base[11, 0], base[11, 1]))
    condensed_fine_tuned_last.append((fine_tuned[11,0], fine_tuned[11,1]))
    condensed_augmented_last.append((augmented[11,0], augmented[11,1]))
    plt.scatter(base[:,0], base[:,1], color="blue", label="Base")
    plt.scatter(fine_tuned[:,0], fine_tuned[:,1], color="green", label="Fine-Tuned")
    plt.scatter(augmented[:,0], augmented[:,1], color="orange", label="Augmented")
    for j, (x, y) in enumerate(base):
        plt.text(x+2, y, f"{j}", fontsize=10)
    for j, (x, y) in enumerate(fine_tuned):
        plt.text(x+2, y, f"{j}", fontsize=10)
    for j, (x, y) in enumerate(augmented):
        plt.text(x+2, y, f"{j}", fontsize=10)
    plt.title(f"t-SNE of MNIST for digit {i}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'../Embedding_Graphs/{dataset_name}/{domain}/Digit_{i}', dpi=600)
    plt.show()

In [ ]:
for i in range(num_classes):
    plt.scatter(condensed_base_last[i][0], condensed_base_last[i][1], label="base", alpha=0.6)
    plt.scatter(condensed_fine_tuned_last[i][0], condensed_fine_tuned_last[i][1], label="fine_tuned", alpha=0.6)
    plt.scatter(condensed_augmented_last[i][0], condensed_augmented_last[i][1], label="augmented", alpha=0.6)
    plt.legend()
    plt.title(f"Digit {i}")
    plt.tight_layout()
    plt.savefig(f'../Embedding_Graphs/{dataset_name}/{domain}/Last_Layer_Digit_{i}', dpi=600)
    plt.show()